# Fine-Tuning Qwen 3.5 0.8B with TRL + SarfTok (Google Colab)

This end-to-end Colab notebook packages **SarfTok** as a wheel, installs it, and then fine-tunes **Qwen 3.5 0.8B** with [TRL](https://github.com/huggingface/trl) using ShareGPT-style Arabic data enriched with SarfTok morphology.


## 0. Session Checklist

1. **Runtime → Change runtime type → GPU (A100/L4 preferred).**
2. **Authenticate with Hugging Face (optional but recommended) to lift rate limits.**
3. **If your SarfTok repo is private, create a temporary PAT or mount Google Drive.**


In [ ]:
# (Optional) Log in to Hugging Face Hub to access gated models or push adapters
from huggingface_hub import notebook_login
notebook_login()


## 1. Clone & Package SarfTok

Replace `YOUR_GITHUB_USERNAME` with the actual path to this repository. The cell builds a wheel via `python -m build` and installs it globally so later steps can `import sarftok` just like any other pip package.


In [ ]:
import os
from pathlib import Path

REPO_URL = "https://github.com/YOUR_GITHUB_USERNAME/sarftokenizer.git"  # TODO: update
WORKDIR = Path("/content")
PKG_DIR = WORKDIR / "sarftokenizer"

if PKG_DIR.exists():
    print("Repository already present, skipping clone.")
else:
    !git clone --depth=1 "$REPO_URL" "$PKG_DIR"

%cd $PKG_DIR
!pip install -q build
!python -m build
wheel_path = sorted((Path("dist").glob("sarftok-*.whl")))[-1]
print("Installing", wheel_path)
!pip install -q "$wheel_path"
%cd /content


## 2. Install Runtime Dependencies

Installs TRL, Transformers, Accelerate, BitsAndBytes (for 4-bit quantization), CAMeL Tools (SarfTok backend), and a recent PyTorch build that Colab ships by default.


In [ ]:
!pip install -q trl transformers accelerate bitsandbytes datasets sentencepiece camel-tools


## 3. Configure Models & Tokenizers

We load Qwen 3.5 0.8B in 4-bit and keep its tokenizer separate from SarfTok's morphological pipeline.


In [ ]:
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer
from trl import SFTTrainer
from peft import LoraConfig
from datasets import load_dataset

from sarftok import SarfTokConfig, SarfTokTokenizer
from sarftok.segmenter import ArabicSegmenter
from sarftok.morph_analyzer.camel_wrapper import CamelMorphAnalyzer

BASE_MODEL = "Qwen/Qwen2.5-0.5B-Instruct"  # Update to Qwen/Qwen2.5-0.8B when available
MAX_SEQ_LEN = 1024
bf16 = torch.bfloat16 if torch.cuda.is_available() else torch.float32

model = AutoModelForCausalLM.from_pretrained(
    BASE_MODEL,
    torch_dtype=bf16,
    device_map="auto",
    load_in_4bit=True,
)
qwen_tokenizer = AutoTokenizer.from_pretrained(BASE_MODEL)
if qwen_tokenizer.pad_token is None:
    qwen_tokenizer.pad_token = qwen_tokenizer.eos_token

sarftok_cfg = SarfTokConfig(
    analyzer_backend="heuristic",
    norm_mode="classical_soft",
    surface_vocab_size=32000,
)
sarftok_tokenizer = SarfTokTokenizer(sarftok_cfg)
sarftok_segmenter = ArabicSegmenter()
sarftok_analyzer = CamelMorphAnalyzer(db_name="calima-msa-r13", top_k=2)


## 4. Build the SarfTok-Enriched Dataset

We pull 3,000 ShareGPT-Arabic chats, summarize their morphology via SarfTok, and store the enriched text in a column named `text` for TRL's `SFTTrainer`.


In [ ]:
DATASET_ID = "FreedomIntelligence/sharegpt-arabic"
MAX_SAMPLES = 3000

raw_dataset = load_dataset(DATASET_ID, split=f"train[:{MAX_SAMPLES}]")


def describe_morph(text: str) -> str:
    words = sarftok_segmenter.segment_flat(text)
    analyses = sarftok_analyzer.analyze_sentence(words, context=text)
    summaries = []
    for word, hyp in zip(words, analyses):
        if not hyp:
            continue
        best = hyp[0]
        parts = []
        if best.root:
            parts.append(f"جذر={best.root}")
        if best.pattern:
            parts.append(f"وزن={best.pattern}")
        if best.pos:
            parts.append(f"نوع={best.pos}")
        if parts:
            summaries.append(f"{word}: " + ", ".join(parts))
    if not summaries:
        return text
    return text + "

### التحليل الصرفي (SarfTok)
" + "
".join(summaries[:32])


def format_dialog(example):
    conv = example.get("conversations") or example.get("messages") or []
    turns = []
    for turn in conv:
        speaker = (turn.get("from") or turn.get("role") or "user").lower()
        tag = "### السؤال" if speaker in {"user", "human"} else "### الإجابة"
        turns.append(f"{tag}
{turn['value'].strip()}")
    text = "

".join(turns)
    return describe_morph(text)

processed = raw_dataset.map(lambda ex: {"text": format_dialog(ex)})
print("Processed records:", len(processed))


## 5. Configure QLoRA (PEFT) + TrainingArguments

We keep rank small to stay within Colab memory and train for 3 epochs over the processed dataset.


In [ ]:
lora_config = LoraConfig(
    r=8,
    lora_alpha=16,
    target_modules=["q_proj", "k_proj", "v_proj", "o_proj", "gate_proj", "down_proj"],
    lora_dropout=0.05,
    bias="none",
    task_type="CAUSAL_LM",
)

from transformers import TrainingArguments

training_args = TrainingArguments(
    output_dir="qwen35_sarftok_trl",
    per_device_train_batch_size=1,
    gradient_accumulation_steps=4,
    learning_rate=2e-4,
    lr_scheduler_type="cosine",
    warmup_ratio=0.05,
    bf16=torch.cuda.is_available(),
    num_train_epochs=3,
    logging_steps=10,
    max_steps=-1,
    save_strategy="epoch",
    report_to="none",
)

trainer = SFTTrainer(
    model=model,
    tokenizer=qwen_tokenizer,
    train_dataset=processed,
    dataset_text_field="text",
    max_seq_length=MAX_SEQ_LEN,
    peft_config=lora_config,
    args=training_args,
)
trainer.train()
trainer.save_model("qwen35_sarftok_trl")


## 6. Test Generation

Load the adapter-weighted model and ask a short medical prompt to verify behavior.


In [ ]:
from transformers import pipeline

peft_model = AutoModelForCausalLM.from_pretrained("qwen35_sarftok_trl", device_map="auto")
peft_tokenizer = qwen_tokenizer

chat = pipeline("text-generation", model=peft_model, tokenizer=peft_tokenizer, max_length=256)
prompt = "### السؤال
اشرح فوائد التعلم العميق في تشخيص الأمراض المزمنة.

### الإجابة
"
print(chat(prompt)[0]['generated_text'])
